In [1]:
transcript = """Interview: Sarah Chen, Finance Operations Manager
Topic: Expense approval process pain points

Sarah: So right now, everything goes through email. Someone submits
a receipt, their manager has to reply-all approve it, then it goes
to finance for a second check if it's over $500. Honestly under
$500 too sometimes, depends who's asking.

Interviewer: How long does that usually take?

Sarah: It varies. Could be same day, could be two weeks if someone's
on leave. We had one sit for three weeks last month because the
manager was on parental leave and nobody knew to escalate it.

Interviewer: What would you want instead?

Sarah: Something with a clear approval chain. If manager doesn't
respond in, I don't know, 48 hours, it should go to someone else.
Maybe their manager? Or we designate a backup approver per team.
Actually backup approver is probably better, we don't want everyone's
boss's boss getting spammed with approvals.

Interviewer: What about the $500 threshold?

Sarah: That should stay, but I want it configurable. Right now it's
basically whatever finance decides that week, which isn't great.
Also — and this is important — travel expenses need a different
flow entirely, they go through a totally separate booking system
already, so this shouldn't touch those. Just regular reimbursements,
office supplies, that kind of thing.

Interviewer: Any reporting needs?

Sarah: Finance needs to see everything pending over $500 at any
time, like a dashboard. Managers just need their own team's stuff.
Oh, and audit trail — we got dinged last year for not being able
to show who approved what and when for a compliance review."""

print(len(transcript), "characters loaded")

1638 characters loaded


In [2]:
from groq import Groq
client = Groq()
print("Groq client ready")

Groq client ready


In [3]:
prompt = f"""You are a Business Analyst assistant. Read the stakeholder interview 
transcript below and extract structured requirements.

Produce:
1. Business Objective (1-2 sentences)
2. In-Scope items
3. Out-of-Scope items (explicitly mentioned as excluded)
4. Functional Requirements (numbered list)
5. Non-Functional Requirements (e.g. audit, reporting)
6. Open Questions / Ambiguities (anything unclear, contradictory, or undecided in the transcript — do not guess or resolve these yourself, flag them)
7. User Stories in the format: As a [role], I want [goal], so that [benefit]

Transcript:
{transcript}
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}]
)
print(response.choices[0].message.content)

**1. Business Objective**  
Implement a streamlined, automated expense‑approval solution that reduces processing delays, enforces a clear, configurable approval hierarchy, and provides audit‑ready visibility for finance while leaving travel‑related expenses untouched.

---

**2. In‑Scope Items**  
- Capture and submit regular (non‑travel) expense receipts (e.g., reimbursements, office supplies).  
- Configurable monetary threshold (default $500) that determines when a second‑level finance check is required.  
- Definition of a primary approver (the employee’s manager) and a backup approver per team.  
- Automatic escalation if the primary approver does not act within a configurable time window (proposed 48 hrs).  
- Finance‑only view/dashboard showing all pending expenses > $500.  
- Manager view showing only expenses submitted by members of their team.  
- Full audit trail capturing “who approved/denied, when, and any comments.”  
- Ability for finance to modify the $500 threshold (or

In [4]:
prompt = f"""You are a Business Analyst assistant. Read the stakeholder interview 
transcript below and extract structured requirements.

CRITICAL RULE: Only include requirements, numbers, or details that are explicitly 
stated or clearly implied in the transcript. Do NOT invent industry-standard 
non-functional requirements (security specs, uptime percentages, performance 
targets, compliance standards, retention periods) unless the stakeholder actually 
mentioned them. If a category (e.g., performance, security) is not addressed in 
the transcript, either omit it entirely or explicitly write "Not discussed — 
recommend clarifying with stakeholder" rather than filling in a plausible-sounding 
default.

Produce:
1. Business Objective (1-2 sentences)
2. In-Scope items
3. Out-of-Scope items (explicitly mentioned as excluded)
4. Functional Requirements (numbered list)
5. Non-Functional Requirements (ONLY those the stakeholder actually raised — e.g. audit trail, reporting access differences. If none were mentioned for a standard category like security or performance, say so explicitly rather than inventing one.)
6. Open Questions / Ambiguities (anything unclear, contradictory, or undecided in the transcript — do not guess or resolve these yourself, flag them)
7. User Stories in the format: As a [role], I want [goal], so that [benefit]

Transcript:
{transcript}
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}]
)
print(response.choices[0].message.content)

**1. Business Objective**  
Create a digital expense‑approval solution that speeds up reimbursements, enforces a clear, configurable approval chain (including backup approvers and escalation), retains an audit trail, and provides role‑based reporting for finance and managers.

---

**2. In‑Scope Items**  
- Submission and approval of **regular reimbursements** (e.g., office supplies).  
- Configurable monetary **threshold** that triggers a second‑level finance review.  
- Definition of a **primary manager approver** and a **backup approver** per team.  
- **Escalation** to the backup approver if the primary manager does not act within a set time‑frame.  
- **Finance dashboard** showing all pending expenses **> $500**.  
- **Manager view** showing pending expenses for their own team.  
- **Audit‑trail** logging who approved each expense and when.  

---

**3. Out‑of‑Scope Items** *(explicitly excluded by the stakeholder)*  
- Any expense flow for **travel expenses** (these are handled i

In [7]:
transcript_2 = """Meeting: Internal Tool Request — Customer Support Ticket Tagging
Attendees: Raj Patel (Support Team Lead), Elena Vasquez (Product Manager)

Raj: We need some way to auto-tag tickets by urgency. Right now agents
manually pick a tag and half the time they just pick "medium" because
it's the default and they're moving fast.

Elena: I think the bigger issue is category, not urgency. We don't
actually know if tickets are billing, technical, or account issues
without reading each one. That's what's slowing down routing.

Raj: Sure, category matters, but urgency is what determines if
someone gets paged at 2am. I've had P1 issues sit for six hours
because they were tagged medium.

Elena: Okay, fair. Can we do both? Auto-tag category AND urgency?

Raj: I mean, ideally yes. But if we're prioritizing, urgency first.
That's the thing causing actual damage.

Elena: I'd push back slightly — category feeds our reporting to
leadership, and that reporting is due monthly regardless. Urgency
tagging helps day to day operations but category is what gets
presented upward.

Raj: Different priorities I guess. Can the system suggest a tag and
let the agent confirm or override it? I don't trust full automation
here, we've been burned before by a bot that auto-closed tickets
incorrectly.

Elena: Agreed, no full automation, human has to confirm. What about
accuracy expectations? Does this need to be like 95% right or is
70% okay if it saves time?

Raj: Honestly I don't know. I'd take anything better than what we
have now, which is basically random.

Elena: We also need this to work with our existing ticket system,
we're not replacing that. It's a plugin or add-on, not a new
platform.

Raj: Right, and one more thing — some tickets come in through email,
some through chat widget, some through the phone system transcript.
Does it need to work on all three or just the main ticket queue?

Elena: Let's start with just the ticket queue. Phone transcripts are
a mess anyway, that's a whole separate problem."""

print(len(transcript_2), "characters loaded")

2006 characters loaded


In [8]:
prompt_2 = f"""You are a Business Analyst assistant. Read the stakeholder interview 
transcript below and extract structured requirements.

CRITICAL RULE: Only include requirements, numbers, or details that are explicitly 
stated or clearly implied in the transcript. Do NOT invent industry-standard 
non-functional requirements (security specs, uptime percentages, performance 
targets, compliance standards, retention periods) unless the stakeholder actually 
mentioned them. If a category is not addressed in the transcript, either omit it 
entirely or explicitly write "Not discussed — recommend clarifying with 
stakeholder" rather than filling in a plausible-sounding default.

Produce:
1. Business Objective (1-2 sentences)
2. In-Scope items
3. Out-of-Scope items (explicitly mentioned as excluded)
4. Functional Requirements (numbered list)
5. Non-Functional Requirements (ONLY those the stakeholder actually raised)
6. Open Questions / Ambiguities (anything unclear, contradictory, or undecided in the transcript — do not guess or resolve these yourself, flag them. If two stakeholders disagree, explicitly note the disagreement and both positions rather than picking a side.)
7. User Stories in the format: As a [role], I want [goal], so that [benefit]

Transcript:
{transcript_2}
"""

response_2 = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt_2}]
)
print(response_2.choices[0].message.content)

**1. Business Objective**  
Enable automatic suggestion of ticket **category** (billing, technical, account) and **urgency** (P1, P2, etc.) tags within the existing ticket‑queue UI so agents can confirm/override them, thereby improving routing speed and the accuracy of leadership‑level reporting.

---

**2. In‑Scope Items**  
- Auto‑suggestion of **category** tag for each ticket.  
- Auto‑suggestion of **urgency** tag for each ticket, with urgency taking priority in the suggestion logic.  
- UI element that shows the suggested tags and allows the agent to **confirm or override** them.  
- Implementation as a **plugin/add‑on** to the current ticketing system (no new platform).  
- Functionality limited to tickets that appear in the **main ticket queue** (i.e., tickets already ingested from email, chat, etc., as they flow into the queue).  

**3. Out‑of‑Scope Items (explicitly mentioned)**  
- **Phone‑system transcript** tagging (deferred as a separate problem).  
- Any form of **full au

In [9]:
import streamlit as st
from groq import Groq

st.set_page_config(page_title="BA Requirements Assistant", layout="wide")

@st.cache_resource
def load_client():
    return Groq()

client = load_client()

st.title("AI Business Requirements Assistant")
st.caption("Paste a stakeholder interview or meeting transcript to generate a structured BRD draft.")

transcript_input = st.text_area("Paste transcript here", height=300)

if st.button("Generate Requirements") and transcript_input:
    with st.spinner("Extracting requirements..."):
        prompt = f"""You are a Business Analyst assistant. Read the stakeholder interview 
transcript below and extract structured requirements.

CRITICAL RULE: Only include requirements, numbers, or details that are explicitly 
stated or clearly implied in the transcript. Do NOT invent industry-standard 
non-functional requirements unless the stakeholder actually mentioned them. If a 
category is not addressed, write "Not discussed — recommend clarifying with 
stakeholder" rather than filling in a plausible-sounding default.

Produce:
1. Business Objective (1-2 sentences)
2. In-Scope items
3. Out-of-Scope items (explicitly mentioned as excluded)
4. Functional Requirements (numbered list)
5. Non-Functional Requirements (ONLY those the stakeholder actually raised)
6. Open Questions / Ambiguities (flag disagreements between stakeholders explicitly, citing both positions rather than resolving them)
7. User Stories in the format: As a [role], I want [goal], so that [benefit]

Transcript:
{transcript_input}
"""
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}]
        )
        result = response.choices[0].message.content

    st.markdown("### Generated Requirements Document")
    st.markdown(result)

    st.download_button("Download as text", result, file_name="requirements_draft.txt")

2026-09-26 12:26:05.782 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-26 12:26:05.786 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-26 12:26:06.045 
  command:

    streamlit run C:\Users\NAAJI\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-26 12:26:06.047 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-26 12:26:06.049 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-26 12:26:06.050 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-26 12:26:06.571 Thread 'Thread-3': missing ScriptRunContext! This warning can be ignored when runnin

In [10]:
import os
print(os.getcwd())

C:\Users\NAAJI\BA_Requirements_Assistant


In [11]:
app_code = '''
import streamlit as st
from groq import Groq

st.set_page_config(page_title="BA Requirements Assistant", layout="wide")

@st.cache_resource
def load_client():
    return Groq()

client = load_client()

st.title("AI Business Requirements Assistant")
st.caption("Paste a stakeholder interview or meeting transcript to generate a structured BRD draft.")

transcript_input = st.text_area("Paste transcript here", height=300)

if st.button("Generate Requirements") and transcript_input:
    with st.spinner("Extracting requirements..."):
        prompt = f"""You are a Business Analyst assistant. Read the stakeholder interview 
transcript below and extract structured requirements.

CRITICAL RULE: Only include requirements, numbers, or details that are explicitly 
stated or clearly implied in the transcript. Do NOT invent industry-standard 
non-functional requirements unless the stakeholder actually mentioned them. If a 
category is not addressed, write "Not discussed \u2014 recommend clarifying with 
stakeholder" rather than filling in a plausible-sounding default.

Produce:
1. Business Objective (1-2 sentences)
2. In-Scope items
3. Out-of-Scope items (explicitly mentioned as excluded)
4. Functional Requirements (numbered list)
5. Non-Functional Requirements (ONLY those the stakeholder actually raised)
6. Open Questions / Ambiguities (flag disagreements between stakeholders explicitly, citing both positions rather than resolving them)
7. User Stories in the format: As a [role], I want [goal], so that [benefit]

Transcript:
{transcript_input}
"""
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}]
        )
        result = response.choices[0].message.content

    st.markdown("### Generated Requirements Document")
    st.markdown(result)

    st.download_button("Download as text", result, file_name="requirements_draft.txt")
'''

with open(r"C:\Users\NAAJI\BA_Requirements_Assistant\app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created at C:\\Users\\NAAJI\\BA_Requirements_Assistant")

app.py created at C:\Users\NAAJI\BA_Requirements_Assistant


In [1]:
import os
import shutil

base_dir = os.getcwd()  # should be C:\Users\NAAJI\BA_Requirements_Assistant
submission_dir = os.path.join(r"C:\Users\NAAJI", "Naji_BA_Requirements_Assistant")

os.makedirs(os.path.join(submission_dir, "Notebook"), exist_ok=True)
os.makedirs(os.path.join(submission_dir, "Screenshots"), exist_ok=True)

shutil.copy("app.py", submission_dir)

print("Folder structure created at:", submission_dir)
print("\nStill need to manually add:")
print("1. BA_Requirements_Assistant_Report.docx (downloaded from this chat)")
print("2. Your .ipynb notebook file into the Notebook/ subfolder")
print("3. Your screenshots (three test cases) into the Screenshots/ subfolder")

Folder structure created at: C:\Users\NAAJI\Naji_BA_Requirements_Assistant

Still need to manually add:
1. BA_Requirements_Assistant_Report.docx (downloaded from this chat)
2. Your .ipynb notebook file into the Notebook/ subfolder
3. Your screenshots (three test cases) into the Screenshots/ subfolder
